In [ ]:
!pip install dash dash-bootstrap-components pyngrok


In [ ]:
import pandas as pd
from dash import Dash, html, dcc, Input, Output
import plotly.express as px
import dash_bootstrap_components as dbc

# Load dataset
df = pd.read_csv("employee_attrition_dataset.csv")


In [ ]:
# Initialize app
app = Dash(__name__,
           external_stylesheets=[dbc.themes.BOOTSTRAP])


In [ ]:
    # App layout
app.layout = dbc.Container([
    html.H1("Employee Dashboard", className="text-center my-4"),

    html.P(
    "This dataset contains detailed information about employees, including their age, gender, marital status, "
    "monthly income, job role, department, and other HR-related attributes. "
    "It is useful for understanding workforce patterns, identifying potential issues related to attrition, "
    "and supporting human resources decisions.\n\n"
    "This dashboard visualizes key aspects of the employee data such as age distribution, gender balance, "
    "marital status, and income spread. It allows filtering by specific values to analyze patterns "
    "and gain insights into the employee structure.",
    className="text-center mb-4"
),


# Row 1: Age Distribution and Gender Distribution
    dbc.Row([
        # Age Distribution
        dbc.Col([
            html.H4("Age Distribution"),
            html.P("This chart shows the distribution of employee ages across different departments. "
                   "You can filter by age to focus on specific employee age groups and understand the age distribution "
                   "in a given range. It helps to analyze age diversity and trends."),
            dcc.Input(
                id='filter-age',
                type='number',
                placeholder='Enter Age...',
                style={'marginBottom': '10px', 'width': '100%'}
            ),
            dbc.Card([
                dcc.Graph(id='graph-age', style={'height': '250px'})
            ], color="danger", outline=False, className="mb-4 shadow border border-danger border-3"),
            html.Div(id='age-description')
        ], md=6),

        # Gender Distribution
        dbc.Col([
            html.H4("Gender Distribution"),
            html.P("This pie chart visualizes the gender distribution among employees. You can filter by gender "
                   "to analyze gender diversity in the organization."),
            dcc.Input(
                id='filter-gender',
                type='text',
                placeholder='Enter Gender...',
                style={'marginBottom': '10px', 'width': '100%'}
            ),
            dbc.Card([
                dcc.Graph(id='graph-gender', style={'height': '250px'})
            ], color="danger", outline=False, className="mb-4 shadow border border-danger border-3"),
            html.Div(id='gender-description')
        ], md=6),
    ]),
    # Row 2: Marital Status and Monthly Income
    dbc.Row([
        # Marital Status Distribution
        dbc.Col([
            html.H4("Marital Status Distribution"),
            html.P("This bar chart shows the distribution of employees' marital status across the company. "
                   "You can filter by gender (Male/Female) to analyze marital status distribution in specific gender groups."),
            dcc.Input(
                id='filter-marital',
                type='text',
                placeholder='Enter Gender (Male or Female)...',
                style={'marginBottom': '10px', 'width': '100%'}
            ),
            dbc.Card([
                dcc.Graph(id='graph-marital', style={'height': '250px'})
            ], color="danger", outline=False, className="mb-4 shadow border border-danger border-3"),
            html.Div(id='marital-description')
        ], md=6),

        # Monthly Income Distribution
        dbc.Col([
            html.H4("Monthly Income Distribution"),
            html.P("This box plot displays the spread of monthly incomes of employees. "
                   "You can filter by job role to observe how income varies across different job positions."),
            dcc.Input(
                id='filter-income',
                type='number',
                placeholder='Enter Monthly Income...',
                style={'marginBottom': '10px', 'width': '100%'}
            ),
            dbc.Card([
                dcc.Graph(id='graph-income', style={'height': '250px'})
            ], color="danger", outline=False, className="mb-4 shadow border border-danger border-3"),
            html.Div(id='income-description')
        ], md=6),
    ]),
], fluid=True)


In [ ]:
# Callback for Age
@app.callback(
    Output('graph-age', 'figure'),
    Output('age-description', 'children'),
    Input('filter-age', 'value')
)
def update_age_graph(value):
    dff = df[df['Age'] == value] if value else df
    fig = px.histogram(dff, x='Age', nbins=20, title='Age Distribution', hover_data=['Age'])
    description = f"Total Employees with Age {value}: {dff.shape[0]}" if value else f"Total Employees: {df.shape[0]}"
    return fig, description


In [ ]:
# Callback for Gender
@app.callback(
    Output('graph-gender', 'figure'),
    Output('gender-description', 'children'),
    Input('filter-gender', 'value')
)
def update_gender_graph(value):
    dff = df[df['Gender'] == value] if value else df
    fig = px.pie(dff, names='Gender', title='Gender Distribution')
    description = f"Gender proportions for selected Gender: {dff['Gender'].value_counts().to_dict()}" if value else f"Gender proportions across all employees: {df['Gender'].value_counts().to_dict()}"
    return fig, description


In [ ]:
@app.callback(
    Output('graph-marital', 'figure'),
    Output('marital-description', 'children'),
    Input('filter-marital', 'value')
)
def update_marital_graph(value):
    if value:
        value = value.strip().upper()
        if value == 'M':
            dff = df[df['Gender'].str.upper() == 'MALE']
        elif value == 'F':
            dff = df[df['Gender'].str.upper() == 'FEMALE']
        else:
            dff = df[0:0]  # Empty DataFrame for invalid input
    else:
        dff = df
    marital_df = dff['Marital_Status'].value_counts().reset_index(name='count')
    marital_df.rename(columns={'index': 'Marital_Status'}, inplace=True)
    fig = px.bar(marital_df, x='Marital_Status', y='count', title='Marital Status Distribution')
    description = f"Marital Status distribution for selected input: {marital_df.to_dict()}" if value else f"Marital Status distribution across all employees: {df['Marital_Status'].value_counts().to_dict()}"
    return fig, description


In [ ]:
# Callback for Income
@app.callback(
    Output('graph-income', 'figure'),
    Output('income-description', 'children'),
    Input('filter-income', 'value')
)
def update_income_graph(value):
    dff = df[df['Monthly_Income'] == value] if value else df
    fig = px.box(dff, x='Monthly_Income', title='Monthly Income Distribution', hover_data=['Monthly_Income'])
    description = f"Employees earning {value} or close are shown." if value else f"Income distribution across all employees."
    return fig, description


In [ ]:
# Run app
if __name__ == '__main__':
    app.run(debug=True)


<IPython.core.display.Javascript object>